# M5-v2 Frame Segmentation Fine-tune (Colab T4)

**목표**: mAP50 0.644 → **0.9+**

**Drive 준비물 (실행 전 업로드 필수):**
1. `frames.zip` — M5 데이터셋 (~280MB)
2. `m5_baseline_best.pt` — 기존 best.pt (207MB)

**업로드 위치는 자유.** 아래 셀 4의 `DRIVE_DIR` 변수만 본인이 올린 폴더 경로로 바꾸면 됨.
- 기본값: `/content/drive/MyDrive/drone_inspect/`
- 다른 예: `/content/drive/MyDrive/`, `/content/drive/MyDrive/colab/m5/`

**예상 시간:** 6~8시간 (T4 16GB, batch=8)

**런타임 설정:** 런타임 → 런타임 유형 변경 → GPU → T4 선택

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. 패키지 설치

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 데이터셋 압축 해제 + baseline 복사

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★★★ 본인이 Google Drive에 frames.zip 과 m5_baseline_best.pt 를 올린 폴더 경로 ★★★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')

# (선택) 파일명도 다르게 올렸다면 여기서 변경
ZIP_NAME = 'frames.zip'
BASELINE_NAME = 'm5_baseline_best.pt'

WORK = Path('/content/m5_train')
WORK.mkdir(parents=True, exist_ok=True)

# 두 파일 모두 존재하는지 먼저 확인
zip_path = DRIVE_DIR / ZIP_NAME
base_path = DRIVE_DIR / BASELINE_NAME
print(f'zip exists?      {zip_path.exists()} -> {zip_path}')
print(f'baseline exists? {base_path.exists()} -> {base_path}')
assert zip_path.exists(), f'{zip_path} 없음 — Drive에 올리고 DRIVE_DIR 경로 확인'
assert base_path.exists(), f'{base_path} 없음 — Drive에 올리고 BASELINE_NAME 확인'

# 데이터셋 unzip — Windows PowerShell Compress-Archive가 만든 zip은
# 경로 separator가 '\'라서 Linux에서 폴더 구조가 인식 안 됨.
# 수동으로 풀면서 '\' → '/' 변환.
ds_dir = WORK / 'frames'
needs_extract = (not ds_dir.exists()) or (not (ds_dir / 'images' / 'val').exists())
if needs_extract:
    # 기존 잘못 풀린 파일 정리 (백슬래시 path 그대로 들어간 파일들)
    for bad in WORK.glob('*\\*'):
        try: bad.unlink()
        except: pass
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)

    print('Extracting frames.zip (Windows-zip safe mode)...')
    with zipfile.ZipFile(zip_path) as z:
        members = z.namelist()
        print(f'  {len(members)} entries')
        for m in members:
            # Windows backslash → POSIX slash
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
else:
    print(f'Already extracted: {ds_dir}')

# baseline best.pt 복사
best_pt = WORK / 'baseline_best.pt'
if not best_pt.exists():
    shutil.copy2(base_path, best_pt)
    print(f'Baseline copied: {best_pt} ({best_pt.stat().st_size/1024/1024:.1f}MB)')

# 데이터셋 구조 확인
print('\n=== 데이터셋 카운트 ===')
for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        cnt = len([f for f in p.glob('*') if f.is_file()])
        print(f'  {split}: {cnt} images')
    else:
        print(f'  {split}: NOT FOUND ({p})')

## 5. data.yaml 생성 (코랩 경로 반영)

In [ ]:
yaml_text = f"""# M5 frame segmentation - Colab
path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: wall_edge
  1: ceiling_edge
  2: door_frame
  3: window_frame
"""
data_yaml = WORK / 'frame_seg.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## 6. Fine-tune 실행 (T4 batch=8, imgsz=960)

In [ ]:
from ultralytics import YOLO

model = YOLO(str(best_pt))

results = model.train(
    data=str(data_yaml),
    epochs=25,
    batch=8,              # T4 16GB → batch=8 가능 (RTX 5070 8GB는 batch=4)
    imgsz=960,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-4,             # 0.9+ 위해 보수적이지 않게
    lrf=0.01,
    patience=8,
    warmup_epochs=2,
    close_mosaic=15,
    freeze=10,            # 백본 안정화
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=3.0, translate=0.1, scale=0.3,
    shear=1.0, perspective=0.0005,
    flipud=0.0, fliplr=0.5,
    mosaic=0.8, mixup=0.0,
    erasing=0.0,
    copy_paste=0.2,       # 약한 증강 (엣지 보존)
    multi_scale=0.2,
    save_period=5,
    plots=True,           # 코랩에서는 plot 보기 좋음
    project='/content/runs',
    name='m5_finetune',
    exist_ok=True,
)

print('Train done.')

## 7. ONNX Export

In [ ]:
best_path = Path('/content/runs/m5_finetune/weights/best.pt')
print(f'best.pt: {best_path} ({best_path.stat().st_size/1024/1024:.1f}MB)')

best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)

onnx_path = best_path.with_suffix('.onnx')
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f}MB)')

## 8. 평가 (val set mAP)

In [ ]:
metrics = best_model.val(data=str(data_yaml), imgsz=960, batch=8)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall: {metrics.box.mr:.4f}')

## 9. 결과를 Google Drive에 저장 (다운로드용)

In [ ]:
OUT = DRIVE_DIR / 'm5v2_results'
OUT.mkdir(parents=True, exist_ok=True)

# ONNX 결과
shutil.copy2(onnx_path, OUT / 'm5_yolo_seg_frames.onnx')
shutil.copy2(best_path, OUT / 'm5_v2_best.pt')

# 학습 로그
results_csv = best_path.parent.parent / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, OUT / 'results.csv')

# 학습 plot
for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)

print('Saved to:', OUT)
for f in OUT.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)')

## 10. 끝!

**다음 단계 (로컬 PC에서):**

1. Google Drive `drone_inspect/m5v2_results/m5_yolo_seg_frames.onnx` 다운로드
2. 로컬 `backend/models_weights/m5_yolo_seg_frames.onnx`에 덮어쓰기
3. mAP50 0.9+ 달성 여부 확인 (results.csv 마지막 줄)